# Configuration

In [0]:
# Feature 1.4: Clean and validate the Toronto Airbnb Bronze data.
#
# Bronze preserves the source as received. Silver applies explicit rules:
# cast types, validate identifiers, clean prices, and remove duplicates.

from pyspark.sql import functions as F
from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

from src.ingestion.silver import clean_price

# Real Databricks environment values.
CATALOG = "trippilot_dev"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"

# Source details for this one-city Silver run.
CITY = "toronto"
SNAPSHOT_DATE = "2026-09-16"

# Input and output tables.
BRONZE_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.airbnb_listings_bronze"
SILVER_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.airbnb_listings_silver"
QUARANTINE_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.airbnb_listings_quarantine"

print(f"Bronze input: {BRONZE_TABLE}")
print(f"Silver output: {SILVER_TABLE}")
print(f"Quarantine output: {QUARANTINE_TABLE}")

# Read only the Toronto Bronze slice

In [0]:
# Read only the Toronto snapshot from Bronze.
#
# Filtering by city and snapshot prevents Silver from accidentally
# processing data from another city or another source date.

bronze_df = (
    spark.table(BRONZE_TABLE)
    .filter(
        (F.col("_city") == CITY)
        & (F.col("_snapshot_date") == SNAPSHOT_DATE)
    )
)

bronze_row_count = bronze_df.count()

print(f"Bronze rows read: {bronze_row_count}")
display(bronze_df.limit(5))

# Select and rename business columns

In [0]:
# Select only the columns needed for the trip assistant and future metrics.
#
# Keep Bronze provenance columns so every Silver row remains traceable.

silver_base_df = bronze_df.select(
    F.col("id").alias("listing_id"),
    F.col("listing_url").alias("listing_url"),
    F.col("name").alias("listing_name"),
    F.col("host_id").alias("host_id"),
    F.col("host_name").alias("host_name"),
    F.col("neighbourhood_cleansed").alias("neighbourhood"),
    F.col("latitude").alias("latitude_raw"),
    F.col("longitude").alias("longitude_raw"),
    F.col("room_type").alias("room_type"),
    F.col("price").alias("price_raw"),
    F.col("minimum_nights").alias("minimum_nights_raw"),
    F.col("availability_365").alias("availability_365_raw"),
    F.col("last_review").alias("last_review_raw"),
    F.col("_source").alias("source"),
    F.col("_city").alias("city"),
    F.col("_snapshot_date").alias("snapshot_date"),
    F.col("_raw_path").alias("raw_path"),
    F.col("_run_id").alias("bronze_run_id"),
    F.col("_ingested_at_utc").alias("bronze_ingested_at_utc"),
)

display(silver_base_df.limit(5))

# Clean and cast fields

In [0]:
# Apply explicit Silver transformations.
#
# - Trim text fields.
# - Cast coordinates and numeric fields.
# - Use the reusable Python helper for price cleaning.
# - Keep invalid prices as null rather than guessing.

silver_df = (
    silver_base_df
    .withColumn(
        "listing_id",
        F.trim(F.col("listing_id")),
    )
    .withColumn(
        "listing_name",
        F.trim(F.col("listing_name")),
    )
    .withColumn(
        "neighbourhood",
        F.trim(F.col("neighbourhood")),
    )
    .withColumn(
        "room_type",
        F.trim(F.col("room_type")),
    )
    .withColumn(
        "latitude",
        F.col("latitude_raw").cast(DoubleType()),
    )
    .withColumn(
        "longitude",
        F.col("longitude_raw").cast(DoubleType()),
    )
    .withColumn(
        "minimum_nights",
        F.col("minimum_nights_raw").cast(IntegerType()),
    )
    .withColumn(
        "availability_365",
        F.col("availability_365_raw").cast(IntegerType()),
    )
    .withColumn(
        "price",
        F.udf(clean_price, DoubleType())(F.col("price_raw")),
    )
    .withColumn(
        "last_review",
        F.to_date(F.col("last_review_raw"), "yyyy-MM-dd"),
    )
)

display(silver_df.limit(5))

# Separate valid and invalid records

In [0]:
# Define the minimum requirements for a valid Silver listing.
#
# A listing must have:
# - A non-empty listing ID
# - A non-null price
# - Valid latitude and longitude

valid_df = silver_df.filter(
    F.col("listing_id").isNotNull()
    & (F.trim(F.col("listing_id")) != "")
    & F.col("price").isNotNull()
    & F.col("latitude").isNotNull()
    & F.col("longitude").isNotNull()
)

# Quarantine records that fail validation.
# Do not silently delete them; keep them for investigation.
quarantine_df = silver_df.subtract(valid_df)

valid_row_count = valid_df.count()
quarantine_row_count = quarantine_df.count()

print(f"Valid Silver rows: {valid_row_count}")
print(f"Quarantined rows: {quarantine_row_count}")

display(quarantine_df.limit(10))

# Remove Duplicate

In [0]:
# Remove duplicate listing IDs.
#
# For this one-city, one-snapshot load, keep the first occurrence.
# This is deterministic and easy to explain in a portfolio project.

silver_deduplicated_df = valid_df.dropDuplicates(["listing_id"])

silver_row_count = silver_deduplicated_df.count()

print(f"Silver rows after deduplication: {silver_row_count}")

# Write Silver and quarantine tables

In [0]:
# Write valid Silver records.
#
# For this one-city feature, overwrite only the Silver table.
# Feature 1.7 will introduce registry-driven, multi-city-safe behavior.

(
    silver_deduplicated_df
    .write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

# Write quarantined records separately.
# This preserves visibility into rejected data instead of hiding it.
(
    quarantine_df
    .write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(QUARANTINE_TABLE)
)

print(f"Silver table written: {SILVER_TABLE}")
print(f"Quarantine table written: {QUARANTINE_TABLE}")

# Verify Silver output

In [0]:
# Verify the Silver output.
#
# Checks:
# - No duplicate listing IDs
# - No null listing IDs
# - Prices are numeric or null
# - Coordinates are numeric or null
# - City and snapshot context remain present

verification_df = spark.table(SILVER_TABLE).agg(
    F.count("*").alias("silver_row_count"),
    F.countDistinct("listing_id").alias("distinct_listing_ids"),
    F.count(F.when(F.col("listing_id").isNull(), 1)).alias("null_listing_ids"),
    F.count(F.when(F.col("price").isNull(), 1)).alias("null_prices"),
    F.count(F.when(F.col("latitude").isNull(), 1)).alias("null_latitudes"),
    F.count(F.when(F.col("longitude").isNull(), 1)).alias("null_longitudes"),
)

display(verification_df)

# There must be no duplicate listing IDs in Silver.
silver_count = spark.table(SILVER_TABLE).count()
distinct_count = spark.table(SILVER_TABLE).select("listing_id").distinct().count()

assert silver_count == distinct_count, (
    f"Duplicate listing IDs found: {silver_count} rows vs {distinct_count} IDs"
)

print("Silver verification passed.")